In [ ]:

import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt

from scipy.signal import butter, filtfilt, medfilt, correlate

In [1]:
import numpy as np

In [2]:
from seismic.loading_and_qc import load_segy, qc_summary, remove_dead_bad_traces
from seismic.wiener_deconvolution import deconvolve_all_traces
from seismic.static_correcion import water_column_static
from seismic.signal_processing import butterworth_filter, median_filter_despike, spherical_divergence_correction, AGC, balance_amplitude
from seismic.velocity_field import semblance_panel, run_next_key, InteractiveVelocityPicker
from seismic.velocity_field import finished as CollectorPicked
from seismic.nmo_correction_and_stacking import nmo_correction, nmo_stack_full_line, build_velocity_field
from seismic.plotting import plot_shot_gather, plot_common_offset_gather, plot_first_and_last_shot, plot_first_and_last_common_offset, plot_decon_qc, plot_velocity_field, plot_cmp_gather, plot_full_stacking

In [ ]:
FILEPATH = 'input/seismic.segy'

In [ ]:
data, headers, dt = load_segy(FILEPATH)
t_axis = np.arange(data.shape[1]) * dt

In [ ]:
headers

In [ ]:
qc_summary(data, headers, dt)

In [ ]:
data_clean, bad = remove_dead_bad_traces(data, energy_threshold_factor = 0.1)

In [ ]:
bad.shape

In [ ]:
plot_shot_gather(data_clean, headers, dt, ffid=3)


# Single common-offset gather, nearest to 300 m
plot_common_offset_gather(data_clean, headers, dt, offset_value=3237)


# First vs last shot gather side by side
plot_first_and_last_shot(data_clean, headers, dt)


# Nearest vs farthest common-offset gather side by side
plot_first_and_last_common_offset(data_clean, headers, dt)


In [ ]:
traces_dc = deconvolve_all_traces(data_clean, 0.12, 0.3, 2.5, dt, data_clean.shape[1], 0.001)

In [ ]:
plot_decon_qc(data_clean, traces_dc, headers, dt, headers["ffid"][0])

In [ ]:
plot_decon_qc(data_clean, traces_dc, headers, dt, headers["ffid"][-1])

In [ ]:
data_filtered = butterworth_filter(traces_dc, dt, low_hz=3, high_hz=80, order=4)
data_filtered = median_filter_despike(data_filtered, kernel_size=5)

In [ ]:
plot_shot_gather(data_filtered, headers, dt, ffid=3)


# Single common-offset gather, nearest to 300 m
plot_common_offset_gather(data_filtered, headers, dt, offset_value=3237)


# First vs last shot gather side by side
plot_first_and_last_shot(data_filtered, headers, dt)


# Nearest vs farthest common-offset gather side by side
plot_first_and_last_common_offset(data_filtered, headers, dt)


In [ ]:
data_spheric = spherical_divergence_correction(data_filtered, t_axis, dt, power = 2, v_rms = 2)
data_acg = AGC(data_filtered, dt, window = 0.5)

In [ ]:
plot_shot_gather(data_spheric, headers, dt, ffid=3)


# Single common-offset gather, nearest to 300 m
plot_common_offset_gather(data_spheric, headers, dt, offset_value=3237)


# First vs last shot gather side by side
plot_first_and_last_shot(data_spheric, headers, dt)


# Nearest vs farthest common-offset gather side by side
plot_first_and_last_common_offset(data_spheric, headers, dt)


In [ ]:
plot_shot_gather(data_acg, headers, dt, ffid=3)


# Single common-offset gather, nearest to 300 m
plot_common_offset_gather(data_acg, headers, dt, offset_value=3237)


# First vs last shot gather side by side
plot_first_and_last_shot(data_acg, headers, dt)


# Nearest vs farthest common-offset gather side by side
plot_first_and_last_common_offset(data_acg, headers, dt)


In [ ]:
data_corr = water_column_static(data_spheric, dt, headers["src_depth"], headers["rec_depth"], v_water = 1500)

In [ ]:
plot_shot_gather(data_corr, headers, dt, ffid=3)


# Single common-offset gather, nearest to 300 m
plot_common_offset_gather(data_corr, headers, dt, offset_value=3237)


# First vs last shot gather side by side
plot_first_and_last_shot(data_corr, headers, dt)


# Nearest vs farthest common-offset gather side by side
plot_first_and_last_common_offset(data_corr, headers, dt)


In [ ]:
list_enough_cdp = []
for cdp in np.unique(headers["cdp"]):
    if (headers["cdp"] == cdp).sum() >= 60:
        #print(np.where(headers["cdp"] == cdp)[0].sum())
        list_enough_cdp.append(cdp)

In [ ]:
list_enough_cdp = np.random.choice(list_enough_cdp, 10)

In [ ]:
len(list_enough_cdp)

In [ ]:
full_t0 = np.arange(data.shape[1]) * dt
v_range = np.arange(1400, 4500, 20)
STORE = {}
control_picks = {}
pickers = {}
semblances = {}


for cdp in list_enough_cdp:
    cdp = int(cdp)
    cmp_mask = headers["cdp"] == cdp
    cmp_gather = data_corr[cmp_mask]
    cmp_gather = cmp_gather - np.mean(cmp_gather, axis=1, keepdims=True) # semblance assumes the amplitudes represent reflection energy, not a constant bias
    cmp_gather_amp_corr = balance_amplitude(cmp_gather, dt, win_sec=0.5)
    cmp_offsets = headers["offset"][cmp_mask]
    semblance, t0_axis = semblance_panel(cmp_gather_amp_corr, cmp_offsets, dt, v_range)
    semblances[cdp] = (semblance, t0_axis)

In [ ]:
len(semblances.keys())

In [ ]:
keys_iter = iter(semblances.keys())

In [ ]:
%matplotlib widget
run_next_key(STORE, keys_iter, semblances, full_t0, v_range, on_complete=CollectorPicked)

In [ ]:
%matplotlib inline

In [ ]:
velocity_dict = {
    key: value["vel_int"]
    for key, value in STORE.items()
}

In [ ]:
all_cdps = np.unique(headers["cdp"])
velocity_field = build_velocity_field(velocity_dict, all_cdps, full_t0)

In [ ]:
velocity_field.shape

In [ ]:
plot_velocity_field(velocity_field, all_cdps, full_t0)

In [ ]:
stacked_section, fold_section = nmo_stack_full_line(data_corr, headers, dt, all_cdps, velocity_field, stretch_mute=0.3)

In [ ]:
# --- QC: compare one CDP before/after NMO ---
cdp_to_check = 1072

cmp_mask = headers["cdp"] == cdp_to_check
cmp_gather = data_corr[cmp_mask]
cmp_offsets = headers["offset"][cmp_mask]

row = np.where(all_cdps == cdp_to_check)[0][0]
velocity_t0 = velocity_field[row]

nmo_gather = nmo_correction(cmp_gather, cmp_offsets, dt, velocity_t0, stretch_mute=0.3)

plot_cmp_gather(cmp_gather, nmo_gather, cmp_offsets, dt, title=f"CDP {cdp_to_check}")


In [ ]:
plot_full_stacking(stacked_section, all_cdps, t_axis)

In [ ]:


    # 8. Residual statics (example: align raw CMP traces to the stack, then re-stack)
    shifts = residual_statics_xcorr(nmo_gather, stacked_trace, dt, max_shift_ms=20)
    nmo_gather_corrected = apply_static_shifts(nmo_gather, shifts)
    stacked_trace_final = stack_cmp(nmo_gather_corrected)